In [1]:
import requests
import pandas as pd
pd.set_option('display.max_columns', None)

In [2]:
codes = pd.read_excel('UN_comtrade_codes.xlsx')
codes.head()

,Grondstof,Code
0,Algemeen hout,4407
1,Beuken,440392
2,Fins vuren,440321
3,Eiken,440391
4,OSB Oriented Strand Board,441012


In [7]:
codes.loc[codes.Grondstof == 'Cotton']

,Grondstof,Code


In [ ]:
def get_World_export(code):
        
    API_KEY = ""
    URL = "https://comtradeapi.un.org/data/v1/get/C/A/HS"
    
    params = {
        "period": "2024",
        "cmdCode": f"{code}",        # Cotton (HS chapter 52)
        "flowCode": "X",        # Exports
        "partnerCode": "0",     # World
        "partner2Code" : "0",
        "customsCode": "C00",   # Total / all customs procedures (collapses customs splits)
        "motCode": "0",         # Total / all modes of transport (collapses MOT splits)
        "breakdownMode": "classic",
        "includeDesc": "true",
        "maxRecords": "1000"
    }
    
    headers = {"Ocp-Apim-Subscription-Key": API_KEY}
    
    j = requests.get(URL, params=params, headers=headers).json()
    df = pd.DataFrame(j.get("data", []))
    
    if df.empty:
        raise RuntimeError(j)
    
    # HARD GUARANTEE: one row per exporting country
    out = (df.groupby("reporterISO", as_index=False)["primaryValue"]
             .sum()
             .sort_values("primaryValue", ascending=False)
             .assign(code = code))
    return out

def get_NL_import(code):

    API_KEY = "c4172e3d022d4b34b5d85260e5c63205"
    URL = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

    params2 = {
        "period": "2024",
        "cmdCode": f"{code}",        # Cotton (HS chapter 52)
        "flowCode": "M",        # Imports
        "reporterCode": "528",     # NL
        "customsCode": "C00",   # Total / all customs procedures (collapses customs splits)
        "motCode": "0",         # Total / all modes of transport (collapses MOT splits)
        "breakdownMode": "classic",
        "includeDesc": "true",
        "maxRecords": "1000"
    }

    headers = {"Ocp-Apim-Subscription-Key": API_KEY}

    
    j2 = requests.get(URL, params=params2, headers=headers).json()
    df2 = pd.DataFrame(j2.get("data", []))
    if df2.empty:
        raise RuntimeError(j)
    
    # HARD GUARANTEE: one row per exporting country
    out2 = (df2.groupby("partnerISO", as_index=False)["primaryValue"]
             .sum()
             .sort_values("primaryValue", ascending=False)
             .assign(code = code)).loc[lambda d: d.partnerISO != 'W00']
    return out2



In [10]:
get_NL_import(4407).head(100)

,partnerISO,primaryValue,code
60,SWE,2.967400e+08,4407
17,DEU,2.320246e+08,4407
3,BEL,1.283071e+08,4407
22,FIN,7.904185e+07,4407
23,FRA,5.270341e+07,4407
...,...,...,...
31,HKG,3.052310e+02,4407
27,GMB,2.337940e+02,4407
44,MLT,2.262180e+02,4407
14,COL,2.164800e+01,4407


In [ ]:
get_World_export(52).head()